## <a id='toc1_1_'></a>[Предсказание прочности материала стен](#toc0_)
 <!-- vscode-jupyter-toc-id:0 -->


`Содержание ноутбука`

**Table of contents**<a id='toc0_'></a>    
- [Предсказание прочности материала стен](#toc1_1_)    
- [Предварительная обработка данных](#toc2_)    
  - [Чтение и загрузка данных](#toc2_1_)    
  - [Первичный анализ данных](#toc2_2_)    
  - [Разделение выборки на обучающую и тестовую выборки](#toc2_3_)    
  - [Обработка вещественных признаков (заполнение пропусков)](#toc2_4_)    
  - [Кодирование категориальных признаков](#toc2_5_)    
  - [Детекция выбросов и аномалий в данных](#toc2_6_)    
  - [Подведение итогов раздела 1](#toc2_7_)    
- [Генерация новых признаков](#toc3_)    
  - [Корреляционный анализ входных признаков (построение тепловых карт корреляции)](#toc3_1_)    
  - [Исправление проблемы мультиколлинеарности в данных](#toc3_2_)    
  - [Скалирование данных](#toc3_3_)    
  - [Подведение итогов раздела 2](#toc3_4_)    
- [Выбор моделей ML и метрик](#toc4_)    
- [Обучение моделей ML и подбор гиперпараметров](#toc5_)    
- [Вычисление метрик на новых данных](#toc6_)    
- [Результат работы](#toc7_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

**Объект задачи**: строительная смесь.

**Требование заказчика**: материал высочайшего качества, при этом в наиболее короткие сроки.

**Цель**: найти закономерность, выяснить, крепость стен будет зависеть от смеси или от времени, необходимого на созревание материала

`Формирование окружения для работы`

Работа справедлива для версии `Python 3.14.0`. Версии используемых библиотек зафиксированны и находятся в файле `requirements.txt`. Роли библиотек описаны в таблице. Для воспроизводимости результатов исследования зафиксируем основание генератора случайных чисел, который используется для деления выборки на обучение и тест, а также при обучении моделей.
   


| Наименование библиотеки | Описание                                                                             |
| ----------------------- | ------------------------------------------------------------------------------------ |
| matplotlib              | Библиотека для визуализации данных                                                   |
| pandas                  | Библиотека для работы таблицами и плоскими данными                                   |
| numpy                   | Библиотека для работы с массивами и матрицами                                        |
| scikit-learn            | Библиотека для построения классических моделей машинного обучения и работы с данными |
| scipy                   | Библиотека для работы со статистическими объектами                                   |
| missingno               | Библиотека для визуализации пропусков в данных                                       |
| seaborn                 | Библиотека для визуализации данных                                                   |
| lifelines               | Библиотека для обучения и построения статистических моделей выживаемости             |
| category_encoders       | Библиотека для кодирования категориальных признаков в вещественные значения          |
| typing                  | Библиотека для типизации парамеров функций и переменных                              |
| warnings                | Библиотека для корректировки выводов предупреждений                                  |
| scikit-survival         | Библиотека для построения моделей выживаемости, основанных на методах ML  

In [2]:
import pandas as pd
import numpy as np
import typing as t

In [3]:
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
import warnings
warnings.filterwarnings('ignore')

In [5]:
seed = 13
np.random.seed(seed)

# <a id='toc2_'></a>[Предварительная обработка данных](#toc0_)

Исходный набор данных представляет из себя набор испытаний строительного материала на сжатие. Известно, сколько каждого компонента положили в кубический метр смеси, сколько суток он твердел и какую прочность показал на прессе. Необходимо выявить (предсказать?), от чего больше зависит крепость стен.

`Таблица с семантикой данных, а также типами данных столбцов`

| **№** | **Признак** | **Описание** | **Тип данных признака** |
|---|---|---|---|
| **1** | Cement | Содержание цемента в строительном материале| Вещественный |
| **2** | Blast Furnace Slag |Содержание минеральной добавки в смеси |  Вещественный  |
| **3** | Fly Ash | Содержание минеральной пыли в смеси |  Вещественный  |
| **4** | Water | Количество воды в смеси|  Вещественный  |
| **5** | Superplasticizer | Химическая добавка |  Вещественный  |
| **6** | Coarse Aggregate | Скелет бетона: щебень или гравий | Вещественный |
| **7** | Fine Aggregate | Песок |  Вещественный  |
| **8** | Age | Сколько дней образец твердел до испытания | Целочисленный |
| **9** | Concrete compressive strength |  Какое давление выдерживает образец | Вещественный |

В качестве целевого значения выступает Concrete compressive strength (прочность материала). Стоит упомянуть, что данные не являются независимыми: один и тот же состав могли испытывать в разном возрасте. Компоненты связаны по объёму: вместе они заполняют один кубический метр, поэтому их количества нельзя менять по отдельности, не затронув остальные.

## <a id='toc2_1_'></a>[Чтение и загрузка данных](#toc0_)

### 1.1. <a id='toc1_1_1_'></a>[Загрузка исходных данных](#toc0_)

Благодаря сформированному виртуальному окружению можно приступить к загрузке данных.

Данные представлены в формате .csv: это значит, что нам необходимо воспользоваться средствами библиотеки pandas, чтобы прочитать, распарсить и проанализировать полученный объект.
Воспользуемся методом ```pd.read_csv()``` для того, чтобы прочитать файл с данными и представить его в памяти.

Данный метод принимает на вход:
1. путь до файла, который необходимо прочитать;
2. разделитель (символ, который отделяет разные записи друг от друга);
3. кодировка (кодировка символов в файле);
4. и т.д.

Прочитаем файл при помощи метода и результат положим в переменную data, с которой будем в дальнейшем работать.

Метод вернет объект датафрейма (абстракция над табличными данными и табличными представлениями, предоставляющая возможность реализовывать логику по работе с плоскими данными), который необходимо отобразить после прочтения файла.

Сам датасет представляет из себя набор объектов, каждый из которых является комбинацией сотставных элкментов строительной смеси, а также параметров для проверки прочности различных вариаций цементных смесей. Данные параметры составляют матрицу объект-признак.

In [6]:
data = pd.read_csv('game_of_thrones.csv', sep=';')

### 1.2. <a id='toc2_2_'></a>[Первичный анализ данных](#toc0_)

Первоначально необходимо проанализировать типы данных колонок датафрейма, чтобы правильно представить исходные данные.

Поскольку для дальнейшего моделирования необходимо, чтобы все значения признаков (кроме Age Day) являлись вещественными числами (либо специальным значением NaN, обозначающим нечисловое значение), воспользуемся методом датафрейма ```.info()```, который выводит сводную информацию о колонках объекта. После этого выполним преобразование данных с пропусками, чтобы получить готовый набор данных для работы.

Рассмотрим фрагмент датасета. Первоначально выведем его на экран.

In [7]:
data.head(5)

,Cement kg/m^3,Blast Furnace Slag kg/m^3,Fly Ash kg/m^3,Water kg/m^3,Superplasticizer kg/m^3,Coarse Aggregate kg/m^3,Fine Aggregate kg/m^3,Age day,Concrete compressive strength
0,540.0,0.0,0.0,NaN,2.5,1040.0,676.0,28.0,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28.0,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270.0,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365.0,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360.0,44.30


In [10]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1030 entries, 0 to 1029
Data columns (total 9 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Cement kg/m^3                  968 non-null    float64
 1   Blast Furnace Slag kg/m^3      958 non-null    float64
 2   Fly Ash kg/m^3                 948 non-null    float64
 3   Water kg/m^3                   948 non-null    float64
 4   Superplasticizer kg/m^3        937 non-null    float64
 5   Coarse Aggregate kg/m^3        927 non-null    float64
 6   Fine Aggregate kg/m^3          906 non-null    float64
 7   Age day                        978 non-null    float64
 8   Concrete compressive strength  1030 non-null   float64
dtypes: float64(9)
memory usage: 72.6 KB


## <a id='toc2_3_'></a>[Разделение выборки на обучающую и тестовую выборки](#toc0_)

Для построения статистических моделей, а также их проверки, необходимо сформировать обучающий и тестовый наборы данных.

При помощи метода `train_test_split` сформируем обучающую и тестовую выборки. Поскольку в исходном датасете около 1000 записей, на тест оставим 15% данных. При этом обязательно выполним стратификацию, чтобы и в train-, и в test-выборках сохранялась пропорция цензурированных данных.

Дополнительно, перемешаем данные, чтобы в них не было временных и порядковых зависимостей, которые могли возникнуть при сборе датаета и порядке хранения записей в БД.

In [20]:
from sklearn.model_selection import train_test_split

Необходимо выделить обучающий и тестовый датасеты именно на данном этапе, чтобы не допустить утечки данных.

In [22]:
y_columns = ['Concrete compressive strength']

train_df, test_df = train_test_split(data, shuffle=True, random_state=seed, test_size=0.15)

X_train, X_test = train_df.drop(columns=y_columns), test_df.drop(columns=y_columns)
y_train, y_test = train_df[y_columns], test_df[y_columns]

## <a id='toc2_4_'></a>[Обработка вещественных признаков (заполнение пропусков)](#toc0_)

## <a id='toc2_5_'></a>[Кодирование категориальных признаков](#toc0_)

## <a id='toc2_6_'></a>[Детекция выбросов и аномалий в данных](#toc0_)

## <a id='toc2_7_'></a>[Подведение итогов раздела 1](#toc0_)
<!-- А таких разделах обычно подытоживается вся информация о тех действиях, которые вы делали в разделе -->

# <a id='toc3_'></a>[Генерация новых признаков](#toc0_)

## <a id='toc3_1_'></a>[Корреляционный анализ входных признаков (построение тепловых карт корреляции)](#toc0_)

## <a id='toc3_2_'></a>[Исправление проблемы мультиколлинеарности в данных](#toc0_)

## <a id='toc3_3_'></a>[Скалирование данных](#toc0_)

## <a id='toc3_4_'></a>[Подведение итогов раздела 2](#toc0_)
<!-- А таких разделах обычно подытоживается вся информация о тех действиях, которые вы делали в разделе -->

# <a id='toc4_'></a>[Выбор моделей ML и метрик](#toc0_)

`ячейки для выбора моделей ML`
<!-- Третий раздел самый богатый на писанину. Вам необходимо сначала обосновать выбор моделей, а затем все отобранные модели описать (расписать концептуально, как они работают) -->

`ячейки для выбора метрик`
<!-- И снова писанина: необходимо не только описать, какие метрики вы используете, но привести формулы их вычисления -->

In [14]:
# ячейка для импорта моделей из пула

In [15]:
# ячейки для формирования сеток гиперпараметров

In [16]:
# ячейка для импорта метрик из пула

# <a id='toc5_'></a>[Обучение моделей ML и подбор гиперпараметров](#toc0_)

In [17]:
# ячейки для подбора гиперпараметров и обучения лучших моделей

# <a id='toc6_'></a>[Вычисление метрик на новых данных](#toc0_)

In [18]:
# ячейки для вычисления метрик на лучших моделях и их сопоставление

# <a id='toc7_'></a>[Результат работы](#toc0_)

`ячейка для подведения итогов по всей работе`
<!-- Тут обычно пишется, какая модель показала себя лучше, с какими метриками и рекомендуется ли ее выводить в прод -->